In [ ]:
# Hardware diagnostics (GPU/A100)
import os
import platform
import subprocess

def _run_raw(cmd: list[str]) -> tuple[int, str]:
    p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    return p.returncode, p.stdout

print("Platform:", platform.platform())
print("Python:", platform.python_version())

rc, out = _run_raw(["bash", "-lc", "nvidia-smi"])  # present on most GPU images
print("\n== nvidia-smi ==")
print(out if out.strip() else f"(no output; exit_code={rc})")

try:
    import torch

    print("\nTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("CUDA device count:", torch.cuda.device_count())
        print("CUDA device 0:", torch.cuda.get_device_name(0))
except Exception as e:
    print("\nTorch import failed:", repr(e))


## RAMT Production Pipeline (Remote GPU)

- Scope: ingest → features → train → artifacts
- Output: `results/` and `RAMT_Artifacts_v1.zip`


In [ ]:
# Runtime configuration
import os

# Pin GPU selection when multiple devices exist.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

# Reduce log noise in common libs.
os.environ.setdefault("PYTHONWARNINGS", "ignore")

print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))


In [ ]:
# Install (clean logs, capture all output)
import os
import shlex
import subprocess
from pathlib import Path

def run_cmd(cmd: list[str], *, cwd: Path | None = None, env: dict[str, str] | None = None) -> str:
    """Run a command, stream consolidated output, raise on failure."""
    display = " ".join(shlex.quote(c) for c in cmd)
    print(f"\n$ {display}")
    p = subprocess.run(cmd, cwd=str(cwd) if cwd else None, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    out = p.stdout or ""
    if out.strip():
        print(out)
    if p.returncode != 0:
        raise RuntimeError(f"Command failed (exit_code={p.returncode}): {cmd}")
    return out

repo_root = Path.cwd().resolve()
if not (repo_root / "models").exists() or not (repo_root / "features").exists() or not (repo_root / "requirements.txt").exists():
    raise FileNotFoundError(
        "Notebook must be run from the repository root (expected: models/, features/, requirements.txt). "
        f"Current directory: {repo_root}"
    )

print("Using repo root:", repo_root)

run_cmd(["python", "-m", "pip", "install", "-q", "--upgrade", "pip"])
run_cmd(["python", "-m", "pip", "install", "-q", "-r", "requirements.txt"])


In [ ]:
# Automated data ingestion
from pathlib import Path

run_cmd(["python", "scripts/fetch_nifty200.py"])  # writes into data/raw/

raw_dir = Path("data/raw")
print("\n== data/raw status ==")
if not raw_dir.exists():
    raise FileNotFoundError(f"Missing directory: {raw_dir.resolve()}")

raw_files = sorted([p for p in raw_dir.rglob("*") if p.is_file()])
print(f"Files: {len(raw_files)}")
for p in raw_files[:50]:
    print("-", p.as_posix())
if len(raw_files) > 50:
    print(f"... +{len(raw_files) - 50} more")


In [ ]:
# Causal pipeline execution: feature engineering
# Use the current kernel's interpreter so we don't depend on a `python` shim existing.
run_cmd([sys.executable, "-m", "features.feature_engineering"])


In [ ]:
# Core training
# Initialize walk-forward cross-validation
run_cmd(["python", "models/run_final_2024_2026.py", "--epochs", "3", "--step-size", "21"])


In [ ]:
# Artifact lifecycle management: zip results/ and provide a download path
import sys
import zipfile
from pathlib import Path

results_dir = Path("results")
zip_path = Path("RAMT_Artifacts_v1.zip").resolve()

if not results_dir.exists():
    raise FileNotFoundError(f"Missing directory: {results_dir.resolve()}")

if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for p in sorted(results_dir.rglob("*")):
        if p.is_file():
            zf.write(p, arcname=p.as_posix())

print("Created:", str(zip_path))

in_colab = "google.colab" in sys.modules
if in_colab:
    from google.colab import files  # type: ignore

    files.download(str(zip_path))
else:
    print("Local/remote server path:", str(zip_path))


In [ ]:
# Verification: summarize generated artifacts in results/
from pathlib import Path

results_dir = Path("results")
paths = sorted([p for p in results_dir.rglob("*") if p.is_file()])

print("Results directory:", str(results_dir.resolve()))
print("Files:", len(paths))

def _fmt_bytes(n: int) -> str:
    for unit in ["B", "KB", "MB", "GB"]:
        if n < 1024 or unit == "GB":
            return f"{n:.1f}{unit}" if unit != "B" else f"{n}{unit}"
        n /= 1024
    return f"{n:.1f}GB"

for p in paths:
    try:
        size = _fmt_bytes(p.stat().st_size)
    except Exception:
        size = "?"
    print(f"- {p.as_posix()}  ({size})")

weights = [p for p in paths if p.suffix in {".pt", ".pth"}]
scalers = [p for p in paths if p.suffix in {".joblib", ".pkl"} or "scaler" in p.name.lower()]
csvs = [p for p in paths if p.suffix == ".csv" and ("ranking" in p.name.lower() or "rank" in p.name.lower())]

print("\n== Expected artifact hints ==")
print("Model weights candidates:", [p.name for p in weights])
print("Scaler candidates:", [p.name for p in scalers])
print("Ranking CSV candidates:", [p.name for p in csvs])
